<a href="https://colab.research.google.com/github/Adhiaris/Grokking-Deep-Learning/blob/main/chapter_09_activation_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 9: Modeling Probabilities and Nonlinearities
### Activation Functions

Activation functions serve two purposes: introducing nonlinearity into the network and constraining outputs to interpretable ranges (e.g., probabilities between 0 and 1).

## 1. Why Activation Functions Are Necessary

Without activation functions, stacked linear layers collapse into a single linear transformation. No matter how many layers you add, the network can only learn linear relationships.

In [ ]:
import numpy as np

np.random.seed(0)

W0 = np.random.rand(3, 4)
W1 = np.random.rand(4, 2)
W2 = np.random.rand(2, 1)

x = np.array([[1.0, 0.0, 1.0]])

out_with_activation    = x.dot(W0)
out_with_activation    = (out_with_activation > 0) * out_with_activation
out_with_activation    = out_with_activation.dot(W1)
out_with_activation    = (out_with_activation > 0) * out_with_activation
out_with_activation    = out_with_activation.dot(W2)

out_without_activation = x.dot(W0).dot(W1).dot(W2)

print("Output WITH relu (nonlinear)   :", np.round(out_with_activation, 4))
print("Output WITHOUT relu (linear)   :", np.round(out_without_activation, 4))
print("\nWithout activation the three layers are equivalent to one.")

## 2. Common Hidden-Layer Activation Functions

- **ReLU** (Rectified Linear Unit): zero for negative inputs, identity for positive. Fast and effective.
- **Tanh**: squashes output to [-1, 1]. Centered around 0.
- **Sigmoid**: squashes output to (0, 1). Historically popular, now mostly used at the output.

In [ ]:
import numpy as np

x = np.linspace(-3, 3, 7)

def relu(x):    return np.maximum(0, x)
def tanh(x):    return np.tanh(x)
def sigmoid(x): return 1 / (1 + np.exp(-x))

print(f"{'x':>6} {'ReLU':>8} {'Tanh':>8} {'Sigmoid':>10}")
print("-" * 36)
for xi in x:
    print(f"{xi:>6.2f} {relu(xi):>8.4f} {tanh(xi):>8.4f} {sigmoid(xi):>10.4f}")

## 3. Output-Layer Activation Functions

- **Sigmoid**: for binary classification (output is a probability in [0,1]).
- **Softmax**: for multi-class classification (outputs sum to 1 across all classes).
- **Linear (none)**: for regression tasks where output can be any real number.

In [ ]:
import numpy as np

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

raw_scores = np.array([2.0, 1.0, 0.5, -1.0])
probs = softmax(raw_scores)

print("Raw scores :", raw_scores)
print("Softmax    :", np.round(probs, 4))
print("Sum        :", probs.sum())
print("\nEach value is in (0,1) and all values sum to 1.")

## 4. Backpropagating Through Sigmoid

Activation functions must be differentiable for backpropagation. The derivative of sigmoid is `sigmoid(x) * (1 - sigmoid(x))`. During backpropagation, the gradient is multiplied by this derivative.

In [ ]:
import numpy as np

def sigmoid(x):       return 1 / (1 + np.exp(-x))
def sigmoid_deriv(o): return o * (1 - o)

np.random.seed(1)

streetlights = np.array([[1,0,1],[0,1,1],[0,0,1],[1,1,1],[0,1,1],[1,0,1]], dtype=float)
walk_vs_stop = np.array([[0],[1],[0],[1],[1],[0]], dtype=float)

alpha = 0.1
W0 = 2*np.random.rand(3, 4) - 1
W1 = 2*np.random.rand(4, 1) - 1

print(f"{'Epoch':<8} {'Error':>12}")
print("-" * 22)

for j in range(80):
    l0 = streetlights
    l1 = sigmoid(l0.dot(W0))
    l2 = sigmoid(l1.dot(W1))

    l2_err   = np.sum((l2 - walk_vs_stop) ** 2)
    l2_delta = (l2 - walk_vs_stop) * sigmoid_deriv(l2)
    l1_delta = l2_delta.dot(W1.T) * sigmoid_deriv(l1)

    W1 -= alpha * l1.T.dot(l2_delta)
    W0 -= alpha * l0.T.dot(l1_delta)

    if j % 15 == 0:
        print(f"{j:<8} {l2_err:>12.6f}")

## 5. Activation Derivatives Summary

During backpropagation, the gradient is multiplied by the derivative of the activation at each layer.

In [ ]:
import numpy as np

x = np.array([0.6])

def relu(x):          return np.maximum(0, x)
def relu_deriv(x):    return (x > 0).astype(float)
def sigmoid(x):       return 1 / (1 + np.exp(-x))
def sigmoid_deriv(o): return o * (1 - o)
def tanh(x):          return np.tanh(x)
def tanh_deriv(o):    return 1 - o**2

print(f"{'Function':<12} {'Output':>10} {'Derivative':>12}")
print("-" * 38)
for name, fn, dn in [
    ("ReLU",    relu,    relu_deriv),
    ("Sigmoid", sigmoid, sigmoid_deriv),
    ("Tanh",    tanh,    tanh_deriv),
]:
    out = fn(x)[0]
    der = dn(out)
    print(f"{name:<12} {out:>10.4f} {der:>12.4f}")

## Summary

Activation functions add nonlinearity and control output range. Hidden layers typically use ReLU. Output layers use sigmoid (binary), softmax (multiclass), or linear (regression). Every activation function needs a derivative for backpropagation. Chapter 10 uses these ideas in convolutional neural networks.